# Residual Spot-Check

Reviews a stratified sample of low-confidence listings to answer:
**Is the top-1 L3 assignment correct, even though the margin was small?**

Each sampled listing shows:
- Product name and description
- Top-1 assignment (what we gave it) + score
- Top-2 assignment (what it almost became) + score
- The margin between them

**Decision gate:**
- If top-1 looks right ≥ 70% of the time → accept all top-1, lower threshold, re-publish
- If top-1 looks wrong frequently → proceed with residual clustering

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'product_classifier_utils.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from product_classifier_utils import (
    build_product_text,
    embed_texts_from_cache,
    get_bedrock_client,
    get_snowflake_session,
    stable_text_hash,
)

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
AWS_PROFILE   = "staging.admin"
AWS_REGION    = "us-east-1"
MODEL_ID      = "amazon.titan-embed-text-v1"

SNOWFLAKE_TABLE = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.NEW_L3_CLASSIFICATIONS"
ANCHORS_PATH    = PROJECT_ROOT / "pipeline/taxonomy/l3_taxonomy_anchors.json"
ANCHOR_CACHE    = PROJECT_ROOT / "artifacts/cache/anchor_cache.pkl"
CACHE_V2_PATH   = PROJECT_ROOT / "artifacts/cache/embedding_cache_new.pkl"
CACHE_V1_PATH   = PROJECT_ROOT / "artifacts/cache/embedding_cache.pkl"
CACHE_KEYS_PATH = PROJECT_ROOT / "artifacts/cache/embedding_cache_keys.pkl"

SAMPLE_PER_BUCKET = 50   # listings per L3 per source
RANDOM_SEED       = 42

## 1. Embed Anchors

In [ ]:
with open(ANCHORS_PATH) as f:
    anchors_data = json.load(f)
anchors       = anchors_data["anchors"]
anchor_ids    = [a["id"]          for a in anchors]
anchor_labels = [a["label"]       for a in anchors]
anchor_texts  = [a["description"] for a in anchors]

bedrock = get_bedrock_client(profile_name=AWS_PROFILE, region=AWS_REGION)
with open(ANCHOR_CACHE, "rb") as f:
    anchor_cache = pickle.load(f)

anchor_hashes = [stable_text_hash(t) for t in anchor_texts]
anchor_matrix = embed_texts_from_cache(
    texts=anchor_texts, text_hashes=anchor_hashes, cache=anchor_cache,
    client=bedrock, model_id=MODEL_ID, show_progress=False, max_workers=1,
)
anchor_norms  = np.linalg.norm(anchor_matrix, axis=1, keepdims=True)
anchor_normed = anchor_matrix / np.clip(anchor_norms, 1e-10, None)
print(f"Anchors ready: {anchor_normed.shape}")

## 2. Load Residual Sample from Snowflake

In [ ]:
sf = get_snowflake_session()

residual = sf.sql(f"""
    SELECT
        PRODUCT_ID, SOURCE,
        PRODUCT_NAME, DESCRIPTION,
        PRICING_STATUS_C, LIST_PRICE_C,
        ASSIGNED_NEW_L3_LABEL,
        CONFIDENCE, CONFIDENCE_MARGIN,
        CURRENT_L3, CURRENT_L4
    FROM {SNOWFLAKE_TABLE}
    WHERE IS_LOW_CONFIDENCE = TRUE
""").to_pandas()

print(f"Total residual: {len(residual):,} listings")
print(f"\nBy source:")
print(residual["SOURCE"].value_counts().to_string())
print(f"\nBy assigned L3:")
print(residual["ASSIGNED_NEW_L3_LABEL"].value_counts().to_string())

## 3. Stratified Sample

In [ ]:
# Sample SAMPLE_PER_BUCKET listings per (SOURCE, ASSIGNED_NEW_L3_LABEL) combination
sample = (
    residual
    .groupby(["SOURCE", "ASSIGNED_NEW_L3_LABEL"], group_keys=False)
    .apply(lambda g: g.sample(min(SAMPLE_PER_BUCKET, len(g)), random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

print(f"Sample size: {len(sample):,} listings")
print(sample.groupby(["SOURCE", "ASSIGNED_NEW_L3_LABEL"]).size().to_string())

## 4. Compute Top-1 and Top-2 for Sample

Looks up embeddings from volume 2 first (13GB, no 32GB load needed).
Falls back to volume 1 for any hashes not in volume 2.

In [ ]:
print("Loading volume 2 cache (13GB)...")
with open(CACHE_V2_PATH, "rb") as f:
    cache_v2 = pickle.load(f)
print(f"Volume 2: {len(cache_v2):,} entries")

print("Loading volume 1 key index...")
with open(CACHE_KEYS_PATH, "rb") as f:
    cache_v1_keys = pickle.load(f)
print(f"Volume 1 keys: {len(cache_v1_keys):,}")

In [ ]:
# Compute hashes for the sample
texts  = build_product_text(sample).tolist()
hashes = [stable_text_hash(t) for t in texts]

in_v2       = [h in cache_v2      for h in hashes]
in_v1_only  = [h in cache_v1_keys and not in_v2[i] for i, h in enumerate(hashes)]
in_neither  = [not in_v2[i] and not in_v1_only[i]  for i in range(len(hashes))]

print(f"Sample coverage:")
print(f"  In volume 2:       {sum(in_v2):,}")
print(f"  In volume 1 only:  {sum(in_v1_only):,}")
print(f"  In neither:        {sum(in_neither):,}")

In [ ]:
# Load volume 1 only if needed
cache_v1 = None
if any(in_v1_only):
    print(f"Loading volume 1 for {sum(in_v1_only):,} sample items (32GB — takes a few minutes)...")
    with open(CACHE_V1_PATH, "rb") as f:
        cache_v1 = pickle.load(f)
    print("Volume 1 loaded.")

def get_embedding(h, i):
    if in_v2[i]:      return cache_v2[h]
    if in_v1_only[i]: return cache_v1[h] if cache_v1 else None
    return None

# Build embedding matrix for sample (skip any with no embedding)
valid_mask = [not in_neither[i] for i in range(len(hashes))]
valid_indices = [i for i, v in enumerate(valid_mask) if v]
vecs = np.array([get_embedding(hashes[i], i) for i in valid_indices], dtype=np.float32)

if cache_v1:
    del cache_v1  # free 32GB immediately

norms  = np.linalg.norm(vecs, axis=1, keepdims=True)
vecs_n = vecs / np.clip(norms, 1e-10, None)
del vecs

# Cosine similarity vs all 13 anchors
sim = vecs_n @ anchor_normed.T   # (n_sample, 13)
del vecs_n

top2_indices = np.argsort(sim, axis=1)[:, -2:][:, ::-1]  # top-2 sorted desc
top1_idx     = top2_indices[:, 0]
top2_idx     = top2_indices[:, 1]
top1_scores  = sim[np.arange(len(sim)), top1_idx]
top2_scores  = sim[np.arange(len(sim)), top2_idx]
margins      = top1_scores - top2_scores

print(f"Cosine similarity computed for {len(valid_indices):,} sample listings.")

In [ ]:
# Attach top-1 and top-2 back to the sample dataframe
sample_valid = sample.iloc[valid_indices].copy().reset_index(drop=True)
sample_valid["TOP1_LABEL"]  = [anchor_labels[i] for i in top1_idx]
sample_valid["TOP1_SCORE"]  = top1_scores.round(4)
sample_valid["TOP2_LABEL"]  = [anchor_labels[i] for i in top2_idx]
sample_valid["TOP2_SCORE"]  = top2_scores.round(4)
sample_valid["MARGIN"]      = margins.round(4)

# Agreement between stored assignment and recomputed top-1
sample_valid["AGREES_WITH_STORED"] = (
    sample_valid["TOP1_LABEL"] == sample_valid["ASSIGNED_NEW_L3_LABEL"]
)
print(f"Recomputed top-1 agrees with stored assignment: "
      f"{sample_valid['AGREES_WITH_STORED'].sum():,}/{len(sample_valid):,} "
      f"({sample_valid['AGREES_WITH_STORED'].mean()*100:.1f}%)")

## 5. Review Results

Look at the top-1 vs top-2 assignments. Key questions:
- Does top-1 look correct for the product?
- Is top-2 a plausible alternative, or clearly wrong?
- Are there systematic patterns (e.g. all antibody residuals actually belong in Proteins & Peptides)?

In [ ]:
# Summary by L3 bucket: how often does top-2 compete with top-1?
print("Top-1 vs Top-2 competition by assigned L3:")
comp = (
    sample_valid
    .groupby(["TOP1_LABEL", "TOP2_LABEL"])
    .size()
    .reset_index(name="COUNT")
    .sort_values(["TOP1_LABEL", "COUNT"], ascending=[True, False])
)
with pd.option_context("display.max_rows", 60):
    print(comp.to_string(index=False))

In [ ]:
# Display sample rows for manual review — sorted by L3 then margin
review_cols = ["SOURCE", "TOP1_LABEL", "TOP1_SCORE", "TOP2_LABEL", "TOP2_SCORE",
               "MARGIN", "PRODUCT_NAME", "DESCRIPTION"]

display_df = sample_valid[review_cols].sort_values(["TOP1_LABEL", "MARGIN"]).reset_index(drop=True)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)
display_df

In [ ]:
# Save sample with top-1/top-2 for offline SME review
output_path = PROJECT_ROOT / "artifacts/analysis/residual_spotcheck_sample.csv"
sample_valid[review_cols + ["PRODUCT_ID", "ASSIGNED_NEW_L3_LABEL", "CONFIDENCE_MARGIN"]].to_csv(
    output_path, index=False
)
print(f"Saved: {output_path}")

## 6. Decision: Accept Top-1 or Cluster?

Run this cell only after reviewing the sample above.

**If top-1 looks correct → accept all top-1 and re-publish:**

In [ ]:
# ── OPTION A: Accept all top-1 assignments ────────────────────────────────────
# Uncomment and run if the spot-check shows top-1 is correct.
# This updates the master CSV and re-publishes to Snowflake.

# MASTER_CSV = PROJECT_ROOT / "artifacts/analysis/master_classification_results.csv"
# master = pd.read_csv(MASTER_CSV)
# print(f"Before: {master['IS_LOW_CONFIDENCE'].sum():,} residuals")
# master["IS_LOW_CONFIDENCE"] = False  # accept all top-1 assignments
# master.to_csv(MASTER_CSV, index=False)
# print(f"After:  {master['IS_LOW_CONFIDENCE'].sum():,} residuals")
# print("Run combine_and_publish.py to re-publish to Snowflake.")

In [ ]:
# ── OPTION B: Proceed with residual clustering ────────────────────────────────
# If top-1 assignments look wrong, open l4_residual_clustering.ipynb.
# The master_residual_for_clustering.csv is already saved and ready.
print("Residual for clustering:")
print(PROJECT_ROOT / "artifacts/analysis/master_residual_for_clustering.csv")